In [1]:
!pip -q install -U transformers accelerate safetensors huggingface_hub statsmodels
from google.colab import drive, userdata
drive.mount("/content/drive")
import sys; sys.path.insert(0, "/content/drive/MyDrive/spectra")
from spectra.common import *
setup_dirs(); seed_all()

# make sure Gemma always loads in fp32 with the new transformers version
sp = REPO / "spectra" / "sae.py"; t = sp.read_text()
t = t.replace("torch_dtype=torch.float32", "dtype=torch.float32")
t = t.replace('attn_implementation="eager").to(device).eval()', 'attn_implementation="eager").float().to(device).eval()')
sp.write_text(t)

import torch, time, numpy as np
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN").strip())
from spectra.sae import GemmaReader, load_sae, extract, load_acts
train, val, test, shift = (load_split(s) for s in ["train", "val", "test", "shift"])
LAYERS = CFG["sae_layers"]

reader = GemmaReader(CFG["gemma"], LAYERS, max_len=CFG["max_len"])
assert reader.n_layers == 26 and reader.d_model == 1152, (reader.n_layers, reader.d_model)
assert next(reader.model.parameters()).dtype == torch.float32
saes = {L: load_sae(L, CFG["sae_width"], CFG["sae_l0"], CFG["sae_repo"], reader.d_model)[0] for L in LAYERS}
print(round(torch.cuda.memory_allocated() / 1e9, 2), "GB on GPU")

# --- sanity check ---
texts = test.narrative.sample(32, random_state=SEED).tolist()
t0 = time.time(); cache, mask, _ = reader.run(texts); dt = time.time() - t0
m = mask.bool()
for L in LAYERS:
    h = cache[L].float(); f = saes[L].encode(h); rec = saes[L].decode(f)
    hv, rv = h[m], rec[m]
    fve = 1 - ((hv - rv) ** 2).sum() / ((hv - hv.mean(0)) ** 2).sum()
    l0 = (f[m] > 0).float().sum(-1).mean()
    print(f"L{L}: variance explained={fve:.3f} | mean L0={l0:.1f} | NaN={torch.isnan(h).any().item()}")
n_total = len(train) + len(val) + len(test) + len(shift) + len(test) + CFG["masked_train_n"]
print(f"one batch: {dt:.2f}s -> full extraction estimate ~{dt * n_total / 32 / 60:.0f} min")

Mounted at /content/drive


config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

resid_post/layer_13_width_16k_l0_medium | config: {'hf_hook_point_in': 'model.layers.13.output', 'hf_hook_point_out': 'model.layers.13.output', 'width': 16384, 'model_name': 'google/gemma-3-1b-pt', 'architecture': 'jump_relu', 'l0': 60, 'affine_connection': False, 'type': 'sae'} | tensors: {'b_dec': (1152,), 'b_enc': (16384,), 'threshold': (16384,), 'w_dec': (16384, 1152), 'w_enc': (1152, 16384)}


config.json:   0%|          | 0.00/247 [00:00<?, ?B/s]

resid_post/layer_17_width_16k_l0_medium/(…): reconstructing file:   0%|          |  0.00B /  151MB            

resid_post/layer_17_width_16k_l0_medium/(…): downloading bytes:           |  0.00B            

resid_post/layer_17_width_16k_l0_medium | config: {'hf_hook_point_in': 'model.layers.17.output', 'hf_hook_point_out': 'model.layers.17.output', 'width': 16384, 'model_name': 'google/gemma-3-1b-pt', 'architecture': 'jump_relu', 'l0': 60, 'affine_connection': False, 'type': 'sae'} | tensors: {'b_dec': (1152,), 'b_enc': (16384,), 'threshold': (16384,), 'w_dec': (16384, 1152), 'w_enc': (1152, 16384)}
4.33 GB on GPU
L13: variance explained=0.837 | mean L0=62.9 | NaN=False
L17: variance explained=0.813 | mean L0=60.0 | NaN=False
one batch: 0.90s -> full extraction estimate ~47 min


In [2]:
for name, d in [("test", test), ("shift", shift), ("val", val), ("train", train)]:
    extract(reader, saes, d.narrative.tolist(), ACTS / f"raw_{name}", LAYERS, batch=CFG["batch"])
print("raw extraction done")

raw_test: 2048/8762
raw_test: 4096/8762
raw_test: 6144/8762
raw_test: 8192/8762
raw_test: 8762/8762
raw_shift: 2048/17527
raw_shift: 4096/17527
raw_shift: 6144/17527
raw_shift: 8192/17527
raw_shift: 10240/17527
raw_shift: 12288/17527
raw_shift: 14336/17527
raw_shift: 16384/17527
raw_shift: 17527/17527
raw_val: 2048/5000
raw_val: 4096/5000
raw_val: 5000/5000
raw_train: 2048/40000
raw_train: 4096/40000
raw_train: 6144/40000
raw_train: 8192/40000
raw_train: 10240/40000
raw_train: 12288/40000
raw_train: 14336/40000
raw_train: 16384/40000
raw_train: 18432/40000
raw_train: 20480/40000
raw_train: 22528/40000
raw_train: 24576/40000
raw_train: 26624/40000
raw_train: 28672/40000
raw_train: 30720/40000
raw_train: 32768/40000
raw_train: 34816/40000
raw_train: 36864/40000
raw_train: 38912/40000
raw_train: 40000/40000
raw extraction done


In [3]:
n = CFG["masked_train_n"]
extract(reader, saes, test.narrative_masked.tolist(), ACTS / "masked_test", [13], batch=CFG["batch"])
extract(reader, saes, train.narrative_masked.tolist()[:n], ACTS / "masked_train", [13], batch=CFG["batch"])
print("masked extraction done")

masked_test: 2048/8762
masked_test: 4096/8762
masked_test: 6144/8762
masked_test: 8192/8762
masked_test: 8762/8762
masked_train: 2048/20000
masked_train: 4096/20000
masked_train: 6144/20000
masked_train: 8192/20000
masked_train: 10240/20000
masked_train: 12288/20000
masked_train: 14336/20000
masked_train: 16384/20000
masked_train: 18432/20000
masked_train: 20000/20000
masked extraction done


In [ ]:
for name, d in [("train", train), ("val", val), ("test", test), ("shift", shift)]:
    ntok = load_acts(ACTS / f"raw_{name}", "ntok")
    assert len(ntok) == len(d), name
    print(name, len(d), "| truncated at max_len:", f"{(ntok >= CFG['max_len'] - 1).mean():.1%}")
for name, k in [("masked_test", len(test)), ("masked_train", CFG["masked_train_n"])]:
    assert len(load_acts(ACTS / name, "ntok")) == k, name
X = load_acts(ACTS / "raw_test", "saemax", 13)
print("test SAE matrix", X.shape, "| avg active features per narrative:", round(X.getnnz(1).mean(), 1))

!cd /content/drive/MyDrive/spectra && python -m pytest -q tests